In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("CHAT_MODEL")

In [4]:
model=init_chat_model(MODEL)

In [5]:
from pydantic import BaseModel,Field

In [6]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [11]:
model_with_structure=model.with_structured_output(Movie, include_raw=True)
model_with_structure

{
  raw: _ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15', 'langchain-openai': '1.4.3'}}, profile={'name': 'GPT-5.4 nano', 'release_date': '2026-03-17', 'last_updated': '2026-03-17', 'open_weights': False, 'max_input_tokens': 400000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh']}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000017A105D5A90>, async_client=<openai.resources.chat.completions.completio

In [8]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='**Inception** is a 2010 science-fiction action film directed by **Christopher Nolan**. It blends high-concept “dream engineering” with a heist plot, exploring how memories and guilt shape what people can’t control.\n\n### Basic details\n- **Release year:** 2010  \n- **Genre:** Sci‑fi, thriller, action  \n- **Running time:** ~2 hours 28 minutes  \n- **Director/Writer:** Christopher Nolan  \n- **Main cast:**\n  - **Leonardo DiCaprio** as Dom Cobb\n  - **Joseph Gordon‑Levitt** as Arthur\n  - **Elliot Page** (as played) as Ariadne\n  - **Ken Watanabe** as Saito\n  - **Tom Hardy** as Eames\n  - **Marion Cotillard** as Mal\n  - **Cillian Murphy** as Robert Fischer\n\n### What the movie is about (story overview)\nDom Cobb is a skilled “extractor” who steals information from people’s subconscious using technology that allows shared dreaming. His team enters others’ dreams to retrieve secrets—until they’re offered a chance to clear his criminal record.\n\nInstead of extractio

In [12]:
model_with_structure.invoke("Provide details about the moview Inception")

{'raw': AIMessage(content='{"title":"Inception","year":2010,"director":"Christopher Nolan","rating":8.8}', additional_kwargs={'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 115, 'total_tokens': 143, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EBoZ7OotQpzO59D1cYf2X0mJaWYSS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ff2c9-082c-7ec2-afbf-69b56e2993f9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 115, 'output_tokens': 28, 'total_tokens': 143, 'input_token_details': {'audio': 0, 'cache_r

In [14]:
from typing import Optional
from pydantic import BaseModel, Field, field_validator

class MovieSimple(BaseModel):
    title: str = Field(description="Movie title")
    year: int = Field(description="Release year")
    director: Optional[str] = Field(default=None, description="Director is optional")
    rating: float = Field(ge=0, le=10, description="Rating out of 10")

    @field_validator("title")
    @classmethod
    def title_not_empty(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("title cannot be empty")
        return v

In [15]:
model_with_structure_simple = model.with_structured_output(MovieSimple)

movie = model_with_structure_simple.invoke(
    "Give details about Inception with title, year, optional director, and rating."
)

print(movie)
print(movie.model_dump())

title='Inception' year=2010 director='Christopher Nolan' rating=8.8
{'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}


In [17]:
# Why model_validator(mode="after")?
# - mode="before": runs on raw input data before field parsing
# - mode="after": runs after fields are parsed, so self.year/self.director are typed values
from pydantic import model_validator

class MovieModeDemo(BaseModel):
    title: str
    year: int
    director: Optional[str] = None

    @model_validator(mode="after")
    def check_director_for_recent_movies(self):
        if self.year >= 2000 and not self.director:
            raise ValueError("director is required for movies from year 2000 onward")
        return self

print("mode='after' is used for cross-field checks on parsed values")

mode='after' is used for cross-field checks on parsed values
